In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from utils import sample_streaming, bpsk_acquisition, tracking
import gnss_tools.signals.gps_l1ca as gps_l1ca
import utils
import yaml
import pprint
import pickle

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
local_data_dir = os.path.join(os.path.dirname(os.path.dirname(utils.__file__)), "local-data")
collects_dir = os.path.join(local_data_dir, "collects")
available_experiment_names = sorted(os.listdir(collects_dir))
print("Available experiments:", ", ".join(available_experiment_names))
experiment_name = available_experiment_names[0]  # change index to select different experiment
experiment_dir = os.path.join(collects_dir, experiment_name)
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = os.path.join(experiment_dir, "metadata.yml")
with open(metadata_filepath, "r") as f:
    metadata = yaml.safe_load(f)
available_collects = sorted(metadata["collections"].keys())
print(f"Available collects from {experiment_name}:\n", "\n ".join(available_collects))
collect_id = available_collects[0]  # change index to select different collect
collect_info = metadata["collections"][collect_id]

collect_filepath = os.path.join(experiment_dir, collect_info["filename"])
print(f"Using collect '{collect_id}' from file:\n {collect_filepath}")
sample_params = sample_streaming.SampleParameters.from_dict(metadata["sample_params"])
samp_rate = metadata["samp_rate"]
print(f"Sample rate: {samp_rate / 1e6} MHz")
pprint.pprint(sample_params)

inter_freq_l1_hz = metadata["bands"]["L1"]["inter_freq"]
print(f"L1 Intermediate Frequency: {inter_freq_l1_hz / 1e6} MHz")

In [ ]:
# Load acquisition results from file
acq_results_directory = os.path.join(local_data_dir, "acquisition-results")
os.makedirs(acq_results_directory, exist_ok=True)
acq_results_version_id = "v1"
acq_results_filepath = os.path.join(
    acq_results_directory, f"{collect_id}.{acq_results_version_id}.pkl"
)
with open(acq_results_filepath, "rb") as f:
    acq_results: dict[str, bpsk_acquisition.AcquisitionResult] = pickle.load(f)

acquired_signal_ids = sorted(list(filter(
    lambda sig_id: acq_results[sig_id].signal_detected, acq_results.keys()
)))
print(f"Acquired signals in collect {collect_id}: ")
print(", ".join(acquired_signal_ids))

In [ ]:
buffer_duration_ms = 40
block_duration_ms = 1
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)
block_size_samples = int(samp_rate * block_duration_ms / 1e3)

duration_to_track_ms = 60000
# duration_to_track_ms = 2000
num_blocks_to_track = duration_to_track_ms // block_duration_ms

tracking_signal_ids = acquired_signal_ids
# tracking_signal_ids = ["G01", "G17", "G30"]

GPS_L1_tracking_signal_params = {
    "G{:02d}".format(prn): tracking.TrackingSignalParameters(
        code_seq=1 - 2 * gps_l1ca.get_GPS_L1CA_code_sequence(prn),
        nominal_code_rate_chips_per_sec=gps_l1ca.CODE_RATE,
        carrier_freq_hz=gps_l1ca.CARRIER_FREQ,
    ) for prn in range(1, 33)
}

# Use same tracking loop parameters for all signals
tracking_loop_params = tracking.TrackingLoopParameters(
    DLL_bandwidth_hz=2.0,
    PLL_bandwidth_hz=20.0,
    FLL_bandwidth_hz=50.0,
    update_period_ms=block_duration_ms,
    block_duration_ms=block_duration_ms,
    samp_rate=samp_rate,
    EPL_chip_spacing=0.5
)

tracking_channels = {
    sig_id: tracking.TrackingChannel(
        tracking_loop_params,
        signal_params=GPS_L1_tracking_signal_params[sig_id],
        initial_uptime_seconds=0.0,  # TODO: set proper acquisition epoch uptime
        initial_code_phase_seconds=acq_results[sig_id].acq_code_phase_seconds,
        initial_carrier_phase_cycles=0.0,
        initial_doppler_freq_hz=acq_results[sig_id].acq_doppler_hz,
        output_capacity=num_blocks_to_track,
    )
    for sig_id in tracking_signal_ids
}

In [ ]:
with sample_streaming.FileSampleStream(
        collect_filepath,
        sample_params,
        buffer_size_samples,
        block_size_samples,
    ) as sample_stream:

    sample_block_generator = sample_stream.sample_block_generator()
    
    for i_block, sample_block in enumerate(sample_block_generator):
        if i_block >= num_blocks_to_track:
            break
        print(f"\r Processing block {i_block}...", end="")

        uptime_seconds = i_block * block_duration_ms * 1e-3
        
        # Mixdown to baseband
        phi_IF = 2 * np.pi * inter_freq_l1_hz * (uptime_seconds + tracking_loop_params.block_sample_time_arr)
        sample_block *= np.exp(-1j * phi_IF)

        # Track each signal
        for sig_id, channel in tracking_channels.items():
            channel.process_sample_block(uptime_seconds, sample_block)
        

In [ ]:
# Save tracking results from file
tracking_results_directory = os.path.join(local_data_dir, "tracking-results")
os.makedirs(tracking_results_directory, exist_ok=True)
tracking_results_version_id = "v1"
tracking_results_filepath = os.path.join(
    tracking_results_directory, f"{collect_id}.{tracking_results_version_id}.pkl"
)
all_tracking_outputs = {
    sig_id: channel.outputs for sig_id, channel in tracking_channels.items()
}
with open(tracking_results_filepath, "wb") as f:
    pickle.dump(tracking_loop_params, f)
    pickle.dump(all_tracking_outputs, f)